# Chest X-ray Expert — Training & Evaluation

**What this notebook does:**
1. Pulls the Doctor-Assistant codebase from GitHub
2. Downloads NIH ChestX-ray14 via the Kaggle API (~42 GB, cached to Drive)
3. Trains `build_chest_xray_expert` (DenseNet-121, 14 multi-label pathologies)
4. Evaluates with per-class AUC + clinical metrics
5. Generates a structured radiology report with Grad-CAM localization

**Runtime:** Set to **L4 GPU** (24 GB VRAM). A100 also works — batch_size=64 at 320×320 fits either.  
**Checkpoints** go to Google Drive after every epoch — safe against session timeouts.

## 0 · GPU check + global CUDA setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — change runtime to L4 or A100'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU : {gpu.name}')
print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Global CUDA speed settings — run BEFORE importing any model ──────────────
#
# 1. TF32: Ampere/Ada GPUs (L4, A100) can use 10-bit mantissa for fp32 matmuls.
#    Near-lossless and ~3× faster on large linear layers. Already True in PyTorch
#    defaults but set explicitly to be certain.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# 2. cuDNN benchmark: profile conv kernels once for YOUR exact input shape and
#    cache the fastest variant. Always worth it for fixed-size inputs (320×320).
torch.backends.cudnn.benchmark = True

# 3. float32 matmul precision: 'high' enables TF32 / BF16 internally for matmul
#    ops, aligned with torch.compile expectations.
torch.set_float32_matmul_precision('high')

print('CUDA global flags set.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/doctor_assistant'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints/chest_xray'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/chest_xray14'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
print('Drive mounted.')
print('Checkpoints :', CKPT_DIR)
print('Dataset     :', DATASET_ROOT)

## 1 · Pull codebase from GitHub

In [ ]:
REPO_URL = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
REPO_DIR = '/content/doctor_assistant'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo ready.')

## 2 · Install dependencies

In [ ]:
%%capture
!pip install -q \
    'monai>=1.3' \
    'nibabel>=5.2' \
    'pydicom>=2.4' \
    'SimpleITK>=2.3' \
    'timm>=0.9' \
    'scikit-image>=0.22' \
    'scikit-learn>=1.4' \
    'pydantic>=2.6'
print('Dependencies installed.')

## 3 · Download NIH ChestX-ray14

**One-time:** upload your `kaggle.json` (Kaggle → Account → Create New Token).  
The ~42 GB dataset is stored to Drive; subsequent sessions skip this step entirely.

In [ ]:
IMAGES_DIR = os.path.join(DATASET_ROOT, 'images')
CSV_PATH   = os.path.join(DATASET_ROOT, 'Data_Entry_2017.csv')

if os.path.isfile(CSV_PATH) and os.path.isdir(IMAGES_DIR):
    n_images = len(os.listdir(IMAGES_DIR))
    print(f'Dataset on Drive: {n_images:,} images — skipping download.')
else:
    print('Downloading NIH ChestX-ray14 via Kaggle API...')
    from google.colab import files
    uploaded = files.upload()   # upload kaggle.json when prompted
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    LOCAL_TMP = '/content/nih_tmp'
    !kaggle datasets download -d nih-chest-xrays/data -p {LOCAL_TMP} --unzip

    import glob, shutil
    os.makedirs(IMAGES_DIR, exist_ok=True)
    for src in glob.glob(f'{LOCAL_TMP}/images_*/*.png'):   # flatten subdirs
        shutil.move(src, IMAGES_DIR)
    for f in ['Data_Entry_2017.csv', 'train_val_list.txt', 'test_list.txt']:
        src = os.path.join(LOCAL_TMP, f)
        if os.path.isfile(src):
            shutil.copy(src, DATASET_ROOT)
    shutil.rmtree(LOCAL_TMP, ignore_errors=True)
    print(f'Done. {len(os.listdir(IMAGES_DIR)):,} images saved to Drive.')

## 4 · Config

In [ ]:
BACKBONE      = 'timm:densenet121'  # CheXNet standard backbone
IMAGE_SIZE    = 320                 # 320×320; use 224 if VRAM is tight
BATCH_SIZE    = 64                  # L4 24 GB handles 64 at 320px with AMP
EPOCHS        = 20
LR            = 1e-4
LR_MIN        = 1e-6
WEIGHT_DECAY  = 1e-5
FREEZE_EPOCHS = 2                   # head warm-up: backbone frozen for N epochs
NUM_WORKERS   = 4
VAL_FRACTION  = 0.1
USE_COMPILE   = True                # torch.compile — ~30-50% speedup on PyTorch 2.x

print(f'{BACKBONE} | {IMAGE_SIZE}px | bs={BATCH_SIZE} | {EPOCHS} epochs | compile={USE_COMPILE}')

## 5 · Build expert + data loaders

In [ ]:
from experts.chest_xray import build_chest_xray_expert
from data.chest_xray14 import CHESTXRAY14_LABELS, dataset_stats, load_chest_xray14
from data.dataset import ScanDataset
from core.enums import BodyPart, Modality
from preprocessing.transforms import PreprocessConfig, build_preprocess
from torch.utils.data import DataLoader

# ── Expert ──
expert = build_chest_xray_expert(backbone=BACKBONE, image_size=IMAGE_SIZE, pretrained=True)
n_params = sum(p.numel() for p in expert.parameters()) / 1e6
print(f'Expert: {expert.name}  |  {n_params:.1f}M params')

# ── Preprocessing ──
train_pre = build_preprocess(
    PreprocessConfig(spatial_size=(IMAGE_SIZE, IMAGE_SIZE), in_channels=3, intensity='scale', augment=True),
    train=True,
)
val_pre = build_preprocess(
    PreprocessConfig(spatial_size=(IMAGE_SIZE, IMAGE_SIZE), in_channels=3, intensity='scale', augment=False),
    train=False,
)

# ── Sample lists ──
print('Loading sample lists from CSV...')
train_samples = load_chest_xray14(DATASET_ROOT, split='train', val_fraction=VAL_FRACTION)
val_samples   = load_chest_xray14(DATASET_ROOT, split='val',   val_fraction=VAL_FRACTION)
print(f'Train: {len(train_samples):,}  Val: {len(val_samples):,}')

stats = dataset_stats(train_samples)
print('\nTrain label counts:')
for label, count in sorted(stats.per_label.items(), key=lambda x: -x[1]):
    print(f'  {label:<22} {count:>7,}  ({count/stats.total*100:.1f}%)')

# ── DataLoaders ──
# pin_memory=True: pages tensors in pinned (page-locked) RAM so the CUDA DMA
#   engine can transfer them without a CPU copy. Required for non_blocking=True
#   (set in the trainer) to actually overlap CPU and GPU work.
# persistent_workers=True: keeps worker processes alive between epochs, saving
#   the ~2-5s respawn overhead at the start of every epoch.
# prefetch_factor=4: each worker pre-loads 4 batches ahead; hides IO latency.
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
train_ds = ScanDataset(train_samples, train_pre, modality=Modality.XRAY, body_part=BodyPart.CHEST)
val_ds   = ScanDataset(val_samples,   val_pre,   modality=Modality.XRAY, body_part=BodyPart.CHEST)
train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
print(f'\nBatches/epoch — train: {len(train_loader):,}  val: {len(val_loader):,}')

## 6 · Trainer setup

In [ ]:
from training.trainer import Trainer, TrainConfig
from training.losses import MultiTaskLoss
from evaluation.multilabel_metrics import MultilabelEvaluator

evaluator = MultilabelEvaluator(class_names=list(CHESTXRAY14_LABELS))

cfg = TrainConfig(
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=True,    # AMP: fp16 forward/backward
    grad_clip=1.0,
    monitor='auc',
    checkpoint_dir=CKPT_DIR,
    resume=True,
    cudnn_benchmark=True,    # auto-tune conv kernels for 320×320
    channels_last=True,      # NHWC layout: 20-40% faster on Tensor Cores
    tf32=True,               # TF32 matmuls on L4/A100
)

optimizer = torch.optim.AdamW(expert.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

trainer = Trainer(
    model=expert,
    evaluator=evaluator,
    config=cfg,
    loss_fn=MultiTaskLoss(multilabel=True),
    optimizer=optimizer,
    scheduler=scheduler,
)
print(f'Trainer ready.  device={trainer.device}  AMP={trainer.use_amp}  channels_last={trainer.channels_last}')

## 6b · torch.compile — kernel fusion (~30-50% speedup)

`torch.compile` traces the model once and fuses small ops into larger GPU kernels,
eliminating Python overhead and memory round-trips. The first forward pass is slow
(compilation), every subsequent one is fast.  
Set `USE_COMPILE = False` in cell 4 if you hit a compile error.

In [ ]:
if USE_COMPILE:
    # 'reduce-overhead': optimizes for throughput (many small batches).
    # 'max-autotune': slower compile, best for large fixed-size batches — use on A100.
    compile_mode = 'reduce-overhead'
    print(f'Compiling expert with mode={compile_mode!r} ...')
    # Compile the expert that lives inside the trainer (it was moved to GPU already).
    trainer.model = torch.compile(trainer.model, mode=compile_mode)
    print('Compilation done. First forward pass will be slow (tracing); rest will be fast.')
else:
    print('torch.compile skipped.')

## 6c · Backbone freeze warm-up

Freeze the DenseNet-121 trunk for `FREEZE_EPOCHS` so the randomly-initialised
classification head stabilises before pretrained weights are disturbed.

In [ ]:
def freeze_backbone(model, frozen: bool) -> None:
    # If model is a compiled wrapper, .backbone lives on the original module.
    base = getattr(model, '_orig_mod', model)
    for p in base.backbone.parameters():
        p.requires_grad = not frozen
    status = 'frozen' if frozen else 'unfrozen'
    trainable = sum(p.numel() for p in base.parameters() if p.requires_grad) / 1e6
    print(f'Backbone {status}. Trainable params: {trainable:.1f}M')

freeze_backbone(trainer.model, frozen=True)

## 7 · Train

In [ ]:
history = []

for epoch in range(EPOCHS):
    if epoch == FREEZE_EPOCHS:
        print(f'\n--- Epoch {epoch}: unfreezing backbone ---')
        freeze_backbone(trainer.model, frozen=False)

    train_loss = trainer._train_one_epoch(train_loader)
    metrics    = trainer._validate(val_loader)
    score      = trainer._monitored_score(metrics)

    trainer._save('last.pt', epoch)
    if score > trainer.best_metric:
        trainer.best_metric = score
        trainer._save('best.pt', epoch)
        print(f'  ★ new best AUC {score:.4f} — saved best.pt')

    scheduler.step()

    record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'lr': optimizer.param_groups[0]['lr'],
        **metrics,
    }
    history.append(record)
    trainer._log(record, score)

print(f'\nTraining complete. Best macro AUC: {trainer.best_metric:.4f}')

## 8 · Test set evaluation

In [ ]:
best_ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location=trainer.device)
# Load into the underlying module if compiled.
base_model = getattr(trainer.model, '_orig_mod', trainer.model)
base_model.load_state_dict(best_ckpt['model'])
print(f'Loaded best checkpoint (epoch {best_ckpt["epoch"]}, AUC={best_ckpt["best_metric"]:.4f})')

test_samples = load_chest_xray14(DATASET_ROOT, split='test', val_fraction=VAL_FRACTION)
test_ds      = ScanDataset(test_samples, val_pre, modality=Modality.XRAY, body_part=BodyPart.CHEST)
test_loader  = DataLoader(test_ds, shuffle=False, **loader_kwargs)
print(f'Test samples: {len(test_samples):,}')

from training.losses import _classification_logits
test_eval = MultilabelEvaluator(class_names=list(CHESTXRAY14_LABELS))
device = trainer.device
trainer.model.eval()

with torch.no_grad():
    for x, targets in test_loader:
        x       = x.to(device, non_blocking=True)
        targets = {k: v.to(device, non_blocking=True) for k, v in targets.items()}
        if trainer.channels_last:
            x = x.to(memory_format=torch.channels_last)
        with torch.autocast(device_type='cuda', enabled=trainer.use_amp):
            outputs = trainer.model(x)
        logits = _classification_logits(outputs)
        if logits is not None and 'label' in targets:
            test_eval.update(logits, targets['label'])

test_metrics = test_eval.compute()
print(f"\nTest macro AUC      : {test_metrics['auc']:.4f}")
print(f"Macro sensitivity   : {test_metrics['macro_sensitivity']:.4f}")
print(f"Macro specificity   : {test_metrics['macro_specificity']:.4f}")
print(f"ECE                 : {test_metrics['ece']:.4f}")
print('\nPer-label AUC:')
for label, auc in sorted(test_metrics['auc_per_label'].items(), key=lambda x: -(x[1] or 0)):
    auc_str = f'{auc:.4f}' if auc is not None else ' n/a '
    sens = test_metrics['sensitivity_per_label'].get(label, float('nan'))
    print(f'  {label:<22} AUC={auc_str}  sens={sens:.3f}')

## 9 · Training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_x   = [r['epoch'] for r in history]
train_loss = [r['train_loss'] for r in history]
val_loss   = [r.get('val_loss', float('nan')) for r in history]
auc_curve  = [r.get('auc') or float('nan') for r in history]
lr_curve   = [r.get('lr', 0) for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs_x, train_loss, label='train'); axes[0].plot(epochs_x, val_loss, label='val')
axes[0].set(title='BCE Loss', xlabel='Epoch'); axes[0].legend()
axes[1].plot(epochs_x, auc_curve, color='green')
axes[1].axhline(test_metrics['auc'], color='orange', linestyle='--', label=f'test={test_metrics["auc"]:.3f}')
axes[1].set(title='Val Macro AUC', xlabel='Epoch'); axes[1].legend()
axes[2].plot(epochs_x, lr_curve, color='purple')
axes[2].set(title='LR (cosine)', xlabel='Epoch'); axes[2].set_yscale('log')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curves.png', dpi=120)
plt.show()

## 10 · Sample report + Grad-CAM

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from ingest.loaders import load_scan
from reporting import findings_from_classification, Reporter, GridZoneLocalizer
from explainability import GradCAM

# ── Pick a positive test image ──
positive_samples = [s for s in test_samples if isinstance(s.label, list) and sum(s.label) > 0]
sample = random.choice(positive_samples[:300])
print('Image     :', os.path.basename(sample.path))
true_labels = [CHESTXRAY14_LABELS[i] for i, v in enumerate(sample.label) if v > 0]
print('True labels:', true_labels)

# ── Predict ──
# Use the unwrapped model for predict() — compiled wrapper doesn't expose .predict().
raw_expert = getattr(trainer.model, '_orig_mod', trainer.model)
scan = load_scan(sample.path, modality=Modality.XRAY, body_part=BodyPart.CHEST)
pred = raw_expert.predict(scan)
print('\nModel probs (top 5):')
for k, v in sorted(pred.class_probs.items(), key=lambda x: -x[1])[:5]:
    print(f'  {k:<22} {v:.3f}')

# ── Grad-CAM ──
cam      = GradCAM(raw_expert)
heatmaps = cam.for_labels(scan.data)

# ── Findings ──
findings = findings_from_classification(
    pred.class_probs,
    thresholds=0.5,
    confidence=pred.confidence,
    heatmaps=heatmaps,
    localizer=GridZoneLocalizer(),
)
print(f'\nFindings ({len(findings)}):')
for f in findings:
    print(' ', f.to_facts())

# ── Report ──
if 'ANTHROPIC_API_KEY' not in os.environ:
    print('\nNote: ANTHROPIC_API_KEY not set — using deterministic template.')
    print('Add it in Colab Secrets (key icon, left panel) for LLM-polished prose.')
reporter = Reporter(auto_llm=True)
report   = reporter.report(findings, scan.meta)
print(f'\n=== REPORT (generator: {report.generator}) ===')
print(report.to_text())

# ── Grad-CAM visualisation ──
top_findings = [f for f in findings if f.present][:3]
n_cols = len(top_findings) + 1
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
img_np = scan.data.permute(1, 2, 0).cpu().numpy()
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
cmap = 'gray' if img_np.shape[2] == 1 else None
if cmap:
    img_np = img_np[:, :, 0]
axes[0].imshow(img_np, cmap=cmap); axes[0].set_title('Input X-ray'); axes[0].axis('off')
for ax, finding in zip(axes[1:], top_findings):
    heat = heatmaps.get(finding.label)
    if heat is not None:
        ax.imshow(img_np, cmap=cmap)
        ax.imshow(heat.numpy(), alpha=0.45, cmap='jet', vmin=0, vmax=1)
    ax.set_title(f'{finding.label}\np={finding.probability:.2f}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/gradcam_sample.png', dpi=120)
plt.show()